# Basic Tasks 

In [0]:
%sql
create or replace table cyntexa_dev.sales.sales_raw as
select * from read_files("/Volumes/cyntexa_dev/sales/sales_volume/sales.csv",
format => "csv",
inferSchema => true,
header => true
)

In [0]:
%sql
select * from read_files("/Volumes/cyntexa_dev/sales/sales_volume/shipments.json", format => "json",
inferSchema => true,
multiLine => true)

In [0]:
%sql
SELECT
    shipment_id,
    customer.customer_id AS customer_id,
    customer.name AS customer_name,
    shipping.city AS city,
    shipping.country AS country,
    amount
FROM read_files(
    "/Volumes/cyntexa_dev/sales/sales_volume/shipments.json",
    format => 'json',
    inferSchema => true,
    multiLine => true
);

In [0]:
%sql
create or replace table cyntexa_dev.sales.shipments_json as
select 
shipment_id,
customer.customer_id AS customer_id,
customer.name AS customer_name,
shipping.city AS city,
shipping.country AS country,
amount
from read_files("/Volumes/cyntexa_dev/sales/sales_volume/shipments.json",
format => "json",
inferSchema => true,
multiLine => true);

In [0]:
%sql
DESCRIBE cyntexa_dev.sales.sales_raw;

In [0]:
%sql
describe extended cyntexa_dev.sales.sales_raw

In [0]:
%sql
describe detail cyntexa_dev.sales.sales_raw;

# Intermediate Tasks 

In [0]:
%sql
create or replace table cyntexa_dev.sales.sales_raw_metadata as
select 
*,
_metadata.file_name as file_name,
_metadata.file_path as file_path
from read_files(
    "/Volumes/cyntexa_dev/sales/sales_volume/sales.csv",
format => "csv",
inferSchema => true,
header => true);

In [0]:
%sql
select * from cyntexa_dev.sales.sales_raw_metadata
limit 10

In [0]:
%sql
create or replace table cyntexa_dev.sales.sales_raw_iceberg
using iceberg as
select *
from read_files("/Volumes/cyntexa_dev/sales/sales_volume/sales.csv",
format => "csv",
inferSchema => true,
header => true);

In [0]:
%sql
describe detail cyntexa_dev.sales.sales_raw

In [0]:
%sql
describe detail cyntexa_dev.sales.sales_raw_iceberg

In [0]:
%sql
select file_name,
file_path,
count(*) as records_count
from cyntexa_dev.sales.sales_raw_metadata
group by file_name, file_path
order by records_count desc;

# ADVANCED TASK

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/cyntexa_dev/sales/sales_volume/orders_normal.csv",
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/cyntexa_dev/sales/sales_volume/orders_pipe.csv",
    format => 'csv',
    header => true,
    sep => '|',
    inferSchema => true
);

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/cyntexa_dev/sales/sales_volume/orders_extra_column.csv",
    format => 'csv',
    header => true,
    inferSchema => true
);

# Detect mismatch file format before processing 

## Before processing the files, I would validate the incoming file schema against the expected schema. The validation would check column names, column count, delimiter, and required fields. If an unexpected column, missing column, or incorrect delimiter is detected, the file should be rejected or quarantined instead of being loaded into the production table. The source file name from _metadata.file_name should be logged so that the problematic file can be identified quickly.

# Decision Memo

## Native Delta

### Choose native Delta when:

- Databricks is the primary platform
- Downstream systems work well with Delta
- You want the strongest Databricks-native experience

## Iceberg / Delta UniForm

### Consider Iceberg/UniForm when:

- Multiple platforms consume the same data
- Tools such as Snowflake/Trino need interoperability
- Open table format compatibility is important

In [0]:
%sql
DESCRIBE HISTORY cyntexa_dev.sales.sales_raw_metadata;

In [0]:
%sql
update cyntexa_dev.sales.sales_raw_metadata
set total_amount = 0.00

## Now, we can revert the correct data or legal data with versioning or timestamp.

In [0]:
%sql
restore table cyntexa_dev.sales.sales_raw_metadata to version as of 0

In [0]:
%sql
restore table cyntexa_dev.sales.sales_raw_metadata timestamp as of "2026-08-24T11:19:35.000+00:00";

In [0]:
%sql
select * from cyntexa_dev.sales.sales_raw_metadata